# Notebook 01 — Coleta e Limpeza dos Dados

Este notebook consolida todos os dados brutos em um único dataset limpo, pronto para análise e modelagem.

## Objetivo

Produzir o arquivo `data/processed/dataset_municipios.csv` com:
- **~5.570 linhas** (um por município brasileiro)
- **13 features socioeconômicas** (X)
- **1 variável alvo binária** `alta_violencia` (y)
- **1 chave** `cod_ibge` (para referência)

## Fontes dos dados

| Arquivo | Fonte | Variáveis |
|---------|-------|-----------|
| `censo2022_municipios.csv` | IBGE SIDRA — Censo 2022 | pop_total, urbanização, jovens, renda, desemprego, analfabetismo, esgoto |
| `atlas_brasil_municipios.xlsx` | Atlas Brasil — Censo 2010 | IDHM (4), Gini, % pobres |
| `pib_municipios_sidra.csv` | IBGE SIDRA — Tab. 5938 (2021) | PIB per capita |
| `areas_municipios_2024.xls` | IBGE — Áreas Territoriais | área km² → densidade |
| `sinesp_municipios.xlsx` | Sinesp/MJ (2018–2022) | homicídios dolosos → variável alvo |

> **Nota sobre anos:** As features do Atlas Brasil são do Censo 2010 — não há dados equivalentes no Censo 2022 a nível municipal via SIDRA (Gini e IDHM não são publicados). Isso é uma limitação do projeto e será discutida no relatório.

## 1. Configuração

In [1]:
import unicodedata
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests

# Diretórios
ROOT = Path("..").resolve()
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Diretório raiz:", ROOT)
print("Dados brutos:  ", RAW_DIR)
print("Dados limpos:  ", PROCESSED_DIR)

Diretório raiz: /home/gabriel/Projetos/socio_violence_predictor
Dados brutos:   /home/gabriel/Projetos/socio_violence_predictor/data/raw
Dados limpos:   /home/gabriel/Projetos/socio_violence_predictor/data/processed


## 2. Carregamento dos dados brutos

### 2.1 Censo 2022 — IBGE SIDRA

Dados coletados via API SIDRA pelo script `data/download_sidra.py`. Contém 7 variáveis socioeconômicas do Censo 2022 para todos os municípios brasileiros.

In [2]:
censo = pd.read_csv(RAW_DIR / "censo2022_municipios.csv", dtype={"cod_ibge": str})
# Garantir cod_ibge como string de 7 dígitos
censo["cod_ibge"] = censo["cod_ibge"].str.zfill(7)

print("Shape:", censo.shape)
print("Colunas:", list(censo.columns))
censo.head(3)

Shape: (5570, 8)
Colunas: ['cod_ibge', 'pop_total', 'pop_urbana_pct', 'perc_jovens_15_29', 'renda_per_capita', 'taxa_desemprego', 'taxa_analfabetismo_15', 'perc_esgoto_adequado']


,cod_ibge,pop_total,pop_urbana_pct,perc_jovens_15_29,renda_per_capita,taxa_desemprego,taxa_analfabetismo_15,perc_esgoto_adequado
0,1100015,21494.0,60.3471,22.2108,1210.60,1.6919,9.33,0.4678
1,1100023,96833.0,86.6977,24.9894,1458.57,2.8638,6.25,1.9932
2,1100031,5351.0,53.1863,17.4173,1323.79,1.6309,10.62,1.5760


### 2.2 Atlas Brasil — Censo 2010

Dados baixados via automação do site Atlas Brasil (PNUD/IPEA/FJP). Contém IDHM (geral e 3 sub-índices), Índice de Gini e % de pobres para o Censo 2010.

O Atlas Brasil não exporta o código IBGE diretamente — apenas o nome do município no formato `"Nome (UF)"`. É preciso fazer um *lookup* para obter o código de 7 dígitos.

In [3]:
def normalizar_nome(nome) -> str:
    """Remove acentos, converte para minúsculas e colapsa espaços."""
    if not isinstance(nome, str):
        return ""
    nfkd = unicodedata.normalize("NFKD", nome)
    sem_acento = "".join(c for c in nfkd if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", sem_acento).strip().lower()


def _uf_from_municipio(m: dict) -> str:
    """Extrai sigla da UF de um registro da API IBGE de localidades."""
    try:
        return m["microrregiao"]["mesorregiao"]["UF"]["sigla"]
    except (TypeError, KeyError):
        pass
    try:
        return m["regiao-imediata"]["regiao-intermediaria"]["UF"]["sigla"]
    except (TypeError, KeyError):
        return ""


def carregar_lookup_ibge() -> pd.DataFrame:
    """Busca tabela de referência nome+UF → código IBGE via API IBGE."""
    cache = RAW_DIR / "ibge_municipios_ref.csv"
    if cache.exists():
        return pd.read_csv(cache, dtype={"cod_ibge": str})

    print("Baixando referência de municípios via API IBGE...")
    url = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    data = r.json()

    rows = [
        {
            "cod_ibge": str(m["id"]),
            "nome_ibge": m["nome"],
            "uf": _uf_from_municipio(m),
            "nome_norm": normalizar_nome(m["nome"]),
        }
        for m in data
    ]

    df = pd.DataFrame(rows)
    df.to_csv(cache, index=False)
    print(f"  {len(df)} municípios salvos em cache.")
    return df


# --- Carregar Atlas Brasil ---
atlas_raw = pd.read_excel(RAW_DIR / "atlas_brasil_municipios.xlsx")
atlas_raw.columns = ["territorialidade", "gini", "idhm", "idhm_renda",
                     "idhm_longevidade", "idhm_educacao", "perc_pobres"]

# Remover linha do Brasil (total nacional)
atlas_raw = atlas_raw[atlas_raw["territorialidade"] != "Brasil"].copy()

# Extrair nome e UF do formato "Nome do Município (UF)"
atlas_raw["nome_mun"] = atlas_raw["territorialidade"].str.extract(r"^(.*)\s+\(\w{2}\)$")
atlas_raw["uf"] = atlas_raw["territorialidade"].str.extract(r"\((\w{2})\)$")

# Verificar entradas que não bateram no regex
sem_match = atlas_raw["nome_mun"].isna() | atlas_raw["uf"].isna()
if sem_match.any():
    print(f"Entradas sem match no regex ({sem_match.sum()}):")
    print(atlas_raw[sem_match]["territorialidade"].tolist())

atlas_raw = atlas_raw[~sem_match].copy()
atlas_raw["nome_norm"] = atlas_raw["nome_mun"].apply(normalizar_nome)

# --- Lookup IBGE ---
ibge_ref = carregar_lookup_ibge()

# Join por (nome_norm, uf)
atlas = atlas_raw.merge(
    ibge_ref[["cod_ibge", "nome_norm", "uf"]],
    on=["nome_norm", "uf"],
    how="left",
)

# Verificar cobertura do join
nao_encontrados = atlas["cod_ibge"].isna().sum()
print(f"Total municípios Atlas Brasil: {len(atlas)}")
print(f"Código IBGE não encontrado:    {nao_encontrados}")
if nao_encontrados > 0:
    print("Exemplos não encontrados:")
    print(atlas[atlas["cod_ibge"].isna()][["territorialidade", "nome_norm", "uf"]].head(10))

atlas = atlas[["cod_ibge", "gini", "idhm", "idhm_renda",
               "idhm_longevidade", "idhm_educacao", "perc_pobres"]].copy()
atlas.head(3)

Entradas sem match no regex (3):
[' ', 'Elaboração: Atlas do Desenvolvimento Humano no Brasil. Pnud Brasil, Ipea e FJP, 2022.', 'Fontes: dados do IBGE e de registros administrativos, conforme especificados nos metadados disponíveis disponíveis em: http://atlasbrasil.org.br/acervo/biblioteca.']
Baixando referência de municípios via API IBGE...


  5571 municípios salvos em cache.
Total municípios Atlas Brasil: 5565
Código IBGE não encontrado:    26
Exemplos não encontrados:
                  territorialidade                nome_norm  uf
205   Amparo de São Francisco (SE)  amparo de sao francisco  SE
427            Augusto Severo (RN)           augusto severo  RN
491       Barão de Monte Alto (MG)      barao de monte alto  MG
627            Biritiba-Mirim (SP)           biritiba-mirim  SP
756                Brasópolis (MG)               brasopolis  MG
1607             Dona Eusébia (MG)             dona eusebia  MG
1643     Eldorado dos Carajás (PA)     eldorado dos carajas  PA
1656                     Embu (SP)                     embu  SP
1814                 Florínia (SP)                 florinia  SP
1834     Fortaleza do Tabocão (TO)     fortaleza do tabocao  TO


,cod_ibge,gini,idhm,idhm_renda,idhm_longevidade,idhm_educacao,perc_pobres
0,5200050,0.42,0.708,0.687,0.830,0.622,6.18
1,3100104,0.47,0.689,0.693,0.839,0.563,7.94
2,5200100,0.43,0.689,0.671,0.841,0.579,8.45


### 2.3 PIB per capita — IBGE SIDRA (Tabela 5938)

O PIB per capita municipal está na tabela 5938 do SIDRA (variável 37 = PIB total a preços correntes, em R$ 1.000). Dividimos pelo `pop_total` do Censo 2022 para obter o PIB per capita em R$.

In [4]:
def baixar_pib_sidra(periodo: str = "2021") -> pd.DataFrame:
    """Baixa PIB total municipal (R$ 1.000) via SIDRA Tabela 5938."""
    cache = RAW_DIR / f"pib_municipios_sidra_{periodo}.csv"
    if cache.exists():
        return pd.read_csv(cache, dtype={"cod_ibge": str})

    print(f"Baixando PIB municipal {periodo} via SIDRA...")
    url = (
        f"https://servicodados.ibge.gov.br/api/v3/agregados/5938"
        f"/periodos/{periodo}/variaveis/37?localidades=N6%5Ball%5D"
    )
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    data = r.json()

    rows = []
    for bloco in data[0]["resultados"]:
        for item in bloco["series"]:
            cod = item["localidade"]["id"]
            val = item["serie"].get(periodo)
            try:
                val = float(val) if val not in (None, "-", "...") else None
            except (ValueError, TypeError):
                val = None
            rows.append({"cod_ibge": cod, "pib_total_mil": val})

    df = pd.DataFrame(rows)
    df.to_csv(cache, index=False)
    print(f"  {len(df)} municípios salvos.")
    return df


pib_raw = baixar_pib_sidra("2021")

# PIB per capita = (PIB total em R$ 1.000 × 1.000) / população
# Usamos pop_total do Censo 2022 como denominador
pib = pib_raw.merge(
    censo[["cod_ibge", "pop_total"]],
    on="cod_ibge", how="left"
)
pib["pib_per_capita"] = (pib["pib_total_mil"] * 1_000) / pib["pop_total"]
pib = pib[["cod_ibge", "pib_per_capita"]]

print("Shape:", pib.shape)
print("Nulos:", pib["pib_per_capita"].isna().sum())
pib.head(3)

Baixando PIB municipal 2021 via SIDRA...


  5570 municípios salvos.
Shape: (5570, 2)
Nulos: 0


,cod_ibge,pib_per_capita
0,1100015,34170.791849
1,1100023,33163.219150
2,1100031,44555.036442


### 2.4 Áreas Territoriais — IBGE 2024

Utilizado para calcular a densidade demográfica: `pop_total / área_km²`.

In [5]:
areas = pd.read_excel(RAW_DIR / "areas_municipios_2024.xls",
                      dtype={"CD_MUN": str})
areas = areas[["CD_MUN", "AR_MUN_2024"]].rename(
    columns={"CD_MUN": "cod_ibge", "AR_MUN_2024": "area_km2"}
)

print("Shape:", areas.shape)
print("Nulos:", areas["area_km2"].isna().sum())
areas.head(3)

Shape: (5575, 2)
Nulos: 2


,cod_ibge,area_km2
0,1101203,798.083
1,1101708,831.857
2,1100403,2651.991


### 2.5 Sinesp — Homicídios Dolosos (2018–2022)

O arquivo do Sinesp/MJ possui **uma aba por estado**. Concatenamos todas as abas e agregamos os homicídios de **2022** (ano mais recente e coincide com o Censo 2022) por município.

**Decisão de projeto:** municípios que não aparecem no Sinesp em 2022 são tratados como **zero homicídios** naquele ano — ausência de registro equivale a ausência de ocorrências.

In [6]:
xl_sinesp = pd.ExcelFile(RAW_DIR / "sinesp_municipios.xlsx")
sinesp_raw = pd.concat(
    [xl_sinesp.parse(sheet) for sheet in xl_sinesp.sheet_names],
    ignore_index=True,
)

print("Total de registros (todos os anos):", len(sinesp_raw))
print("Anos disponíveis:", sorted(sinesp_raw["Mês/Ano"].dt.year.unique()))
print("Municípios únicos:", sinesp_raw["Cód_IBGE"].nunique())

# Filtrar 2022 e agregar por município (soma anual de vítimas)
sinesp_2022 = (
    sinesp_raw[sinesp_raw["Mês/Ano"].dt.year == 2022]
    .groupby("Cód_IBGE", as_index=False)["Vítimas"]
    .sum()
    .rename(columns={"Cód_IBGE": "cod_ibge", "Vítimas": "homicidios_2022"})
)
sinesp_2022["cod_ibge"] = sinesp_2022["cod_ibge"].astype(str).str.zfill(7)

print(f"\nMunicípios com registro em 2022: {len(sinesp_2022)}")
sinesp_2022.head(3)

Total de registros (todos os anos): 294706
Anos disponíveis: [2018, 2019, 2020, 2021, 2022]
Municípios únicos: 5594

Municípios com registro em 2022: 5394


,cod_ibge,homicidios_2022
0,1100015,2
1,1100023,12
2,1100031,2


## 3. Construção da variável alvo (y)

A variável alvo `alta_violencia` é construída em 3 passos:

1. **Calcular a taxa de homicídios por 100 mil habitantes** para cada município
2. **Calcular a mediana nacional** dessa taxa como limiar (threshold)
3. **Binarizar:** `alta_violencia = 1` se taxa > mediana, `0` caso contrário

Esta abordagem produz automaticamente um dataset **balanceado** (50% de cada classe), o que é desejável para modelos de classificação.

> **Atenção:** municípios sem registro no Sinesp recebem 0 homicídios (e portanto taxa 0 = classe 0). Isso pode introduzir viés se estados com dados incompletos concentrarem subnotificações — fato a ser discutido no relatório.

In [7]:
# Base de municípios: todos os presentes no Censo 2022
base = censo[["cod_ibge", "pop_total"]].copy()

# Adicionar homicídios (zero para ausentes)
base = base.merge(sinesp_2022, on="cod_ibge", how="left")
base["homicidios_2022"] = base["homicidios_2022"].fillna(0)

# Taxa por 100 mil habitantes
base["taxa_homicidios"] = (
    base["homicidios_2022"] / base["pop_total"].replace(0, np.nan) * 100_000
)

# Limiar = mediana nacional
limiar = base["taxa_homicidios"].median()
print(f"Mediana da taxa de homicídios: {limiar:.2f} por 100k hab")

# Variável alvo binária
base["alta_violencia"] = (base["taxa_homicidios"] > limiar).astype(int)

print("\nDistribuição da variável alvo:")
print(base["alta_violencia"].value_counts())
print(f"\nBalanceamento: {base['alta_violencia'].mean():.1%} classe 1")

Mediana da taxa de homicídios: 12.03 por 100k hab

Distribuição da variável alvo:
alta_violencia
0    2785
1    2785
Name: count, dtype: int64

Balanceamento: 50.0% classe 1


## 4. Cruzamento dos datasets

Fazemos o join de todos os datasets pela chave `cod_ibge`. O join é do tipo **left** partindo do Censo 2022 (que tem todos os municípios), garantindo que nenhum município seja perdido por ausência em uma fonte específica.

In [8]:
# Começar com variável alvo + pop_total
df = base[["cod_ibge", "pop_total", "alta_violencia", "taxa_homicidios"]].copy()

# Censo 2022 (excluindo pop_total que já está em df)
censo_features = censo.drop(columns=["pop_total"])
df = df.merge(censo_features, on="cod_ibge", how="left")

# Atlas Brasil (Censo 2010)
df = df.merge(atlas, on="cod_ibge", how="left")

# PIB per capita
df = df.merge(pib, on="cod_ibge", how="left")

# Áreas territoriais
df = df.merge(areas, on="cod_ibge", how="left")

# Densidade demográfica
df["densidade_demografica"] = df["pop_total"] / df["area_km2"]

print("Shape final:", df.shape)
print("\nColunas:")
for col in df.columns:
    print(f"  {col}")

Shape final: (5570, 19)

Colunas:
  cod_ibge
  pop_total
  alta_violencia
  taxa_homicidios
  pop_urbana_pct
  perc_jovens_15_29
  renda_per_capita
  taxa_desemprego
  taxa_analfabetismo_15
  perc_esgoto_adequado
  gini
  idhm
  idhm_renda
  idhm_longevidade
  idhm_educacao
  perc_pobres
  pib_per_capita
  area_km2
  densidade_demografica


## 5. Qualidade dos dados

Verificamos a quantidade de valores ausentes por coluna e decidimos a estratégia de imputação.

In [9]:
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(2)
qualidade = pd.DataFrame({"nulos": nulos, "pct": nulos_pct})
qualidade = qualidade[qualidade["nulos"] > 0].sort_values("pct", ascending=False)

print(f"Total de municípios: {len(df)}")
print(f"\nColunas com valores ausentes:")
print(qualidade.to_string())

Total de municípios: 5570

Colunas com valores ausentes:
                      nulos   pct
perc_pobres              33  0.59
gini                     31  0.56
idhm                     31  0.56
idhm_renda               31  0.56
idhm_longevidade         31  0.56
idhm_educacao            31  0.56
taxa_desemprego          29  0.52
perc_esgoto_adequado     25  0.45


### 5.1 Estratégia de imputação

Para valores ausentes nas **features** (X), utilizamos imputação pela **mediana** da coluna. A mediana é preferível à média por ser robusta a outliers — que são comuns em indicadores municipais com grande heterogeneidade (ex.: municípios com PIB per capita muito alto ou muito baixo).

> A imputação definitiva (para os modelos) será feita no notebook `03_preprocessamento.ipynb`. Aqui apenas verificamos e registramos a situação dos dados.

In [10]:
# Colunas de features (excluir cod_ibge, alta_violencia, taxa_homicidios, area_km2)
FEATURES = [
    "pop_total", "pop_urbana_pct", "perc_jovens_15_29", "renda_per_capita",
    "taxa_desemprego", "taxa_analfabetismo_15", "perc_esgoto_adequado",
    "gini", "idhm", "idhm_renda", "idhm_longevidade", "idhm_educacao",
    "perc_pobres", "pib_per_capita", "densidade_demografica",
]

# Remover municípios sem variável alvo (pop = 0 ou taxa_homicidios NaN)
df = df[df["pop_total"] > 0].copy()

# Imputação por mediana nas features
for col in FEATURES:
    if col in df.columns and df[col].isna().any():
        mediana = df[col].median()
        n = df[col].isna().sum()
        df[col] = df[col].fillna(mediana)
        print(f"  {col}: {n} nulos imputados com mediana={mediana:.4f}")

print(f"\nShape após imputação: {df.shape}")
print(f"Nulos restantes: {df[FEATURES].isnull().sum().sum()}")

  taxa_desemprego: 29 nulos imputados com mediana=4.1338
  perc_esgoto_adequado: 25 nulos imputados com mediana=32.5516
  gini: 31 nulos imputados com mediana=0.4900
  idhm: 31 nulos imputados com mediana=0.6650
  idhm_renda: 31 nulos imputados com mediana=0.6540
  idhm_longevidade: 31 nulos imputados com mediana=0.8080
  idhm_educacao: 31 nulos imputados com mediana=0.5600
  perc_pobres: 33 nulos imputados com mediana=18.1000

Shape após imputação: (5570, 19)
Nulos restantes: 0


## 6. Dataset final

Selecionamos as colunas na ordem definitiva e salvamos o dataset processado.

In [11]:
COLUNAS_FINAIS = ["cod_ibge"] + FEATURES + ["taxa_homicidios", "alta_violencia"]
dataset = df[COLUNAS_FINAIS].copy()

# Salvar
out_path = PROCESSED_DIR / "dataset_municipios.csv"
dataset.to_csv(out_path, index=False, encoding="utf-8")

print(f"✓ Dataset salvo em: {out_path}")
print(f"  Shape: {dataset.shape}")
print(f"  Municípios: {len(dataset)}")
print(f"\n  Distribuição da variável alvo:")
print(dataset["alta_violencia"].value_counts().to_string())
print(f"\n  Nulos por coluna:")
print(dataset.isnull().sum()[dataset.isnull().sum() > 0].to_string() or "  Nenhum!")
print(f"\n  Estatísticas descritivas:")
dataset[FEATURES].describe().round(3)

✓ Dataset salvo em: /home/gabriel/Projetos/socio_violence_predictor/data/processed/dataset_municipios.csv
  Shape: (5570, 18)
  Municípios: 5570

  Distribuição da variável alvo:
alta_violencia
0    2785
1    2785

  Nulos por coluna:
Series([], )

  Estatísticas descritivas:


,pop_total,pop_urbana_pct,perc_jovens_15_29,renda_per_capita,taxa_desemprego,taxa_analfabetismo_15,perc_esgoto_adequado,idhm,idhm_renda,idhm_longevidade,idhm_educacao,perc_pobres,pib_per_capita,densidade_demografica
count,5.570000e+03,5570.000,5570.000,5570.000,5570.000,5570.000,5570.000,5570.000,5570.000,5570.000,5570.000,5570.000,5570.000,5570.000
mean,3.645974e+04,69.039,21.567,1211.587,5.102,10.960,37.862,0.659,0.643,0.802,0.559,23.166,34443.666,115.972
std,2.065187e+05,20.137,2.795,493.338,4.065,7.004,32.909,0.072,0.080,0.045,0.093,17.871,39560.523,594.973
min,8.330000e+02,7.331,11.783,288.650,0.088,0.500,0.032,0.418,0.400,0.672,0.207,0.190,5697.281,0.154
25%,5.228000e+03,54.114,19.676,772.390,2.283,5.100,4.191,0.600,0.572,0.770,0.490,7.040,13608.179,11.336
50%,1.106500e+04,70.835,21.515,1182.540,4.134,8.940,32.552,0.665,0.654,0.808,0.560,18.100,24564.556,24.297
75%,2.442725e+04,86.051,23.384,1520.358,6.903,16.468,67.652,0.718,0.707,0.836,0.630,38.425,41549.294,53.626
max,1.145200e+07,100.000,37.393,4299.910,50.972,36.850,99.948,0.862,0.891,0.894,0.825,78.590,919482.368,13416.814
